# Model Training & Comparison — Superimposed PE Prediction

This notebook trains and compares all models:
- **Classical ML**: Logistic Regression, SVM, Random Forest, XGBoost, LightGBM, CatBoost, Decision Tree, KNN, Naïve Bayes, LDA
- **Deep Learning**: MLP, TabNet, FT-Transformer, LSTM, 1D-CNN
- **Ensemble**: Soft Voting, Stacking, Blending, Bayesian Averaging, Rank Averaging

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import EHRPreprocessor
from src.feature_engineering import FeatureEngineer
from src.models.classical_ml import ClassicalMLModels
from src.models.deep_learning import MLPModel, TabNetModel
from src.models.ensemble import StackingEnsemble, VotingEnsemble, BayesianModelAveraging
from src.evaluation import ModelEvaluator, compute_metrics, cross_validate_model
from src.visualization import ResultsVisualizer

SEED   = 42
TARGET = 'superimposed_pe'

## 1. Load & Preprocess Data

In [ ]:
from pathlib import Path
import subprocess

data_path = Path('../data/synthetic/synthetic_pe_cohort.csv')
if not data_path.exists():
    subprocess.run([sys.executable, '../data/synthetic/generate_synthetic_data.py'], check=True)

df = pd.read_csv(data_path)

# Feature engineering
fe = FeatureEngineer(random_state=SEED)
df = fe.add_clinical_features(df)

# Preprocessing
prep = EHRPreprocessor(target_col=TARGET, random_state=SEED)
X, y, feature_names = prep.fit_transform(df)

# Split
X_train, X_val, X_test, y_train, y_val, y_test = EHRPreprocessor.split(
    X, y, test_size=0.20, val_size=0.10, stratify=True, random_state=SEED)
X_tv = np.vstack([X_train, X_val])
y_tv = np.concatenate([y_train, y_val])

print(f'Train+val: {len(y_tv)}  |  Test: {len(y_test)}  |  Features: {X.shape[1]}')
print(f'Test prevalence: {y_test.mean():.1%}')

## 2. Classical ML Models

In [ ]:
factory   = ClassicalMLModels()
ml_models = factory.get_all()
evaluator = ModelEvaluator(bootstrap=True, n_bootstrap=300)

for key, model in ml_models.items():
    print(f'Training {model.name} ...', end=' ')
    try:
        model.fit(X_tv, y_tv)
        y_prob = model.predict_proba(X_test)[:, 1]
        res = evaluator.add(model.name, y_test, y_prob)
        print(f'AUROC={res["roc_auc"]:.3f}  AP={res["average_precision"]:.3f}')
    except Exception as e:
        print(f'FAILED — {e}')

In [ ]:
table_ml = evaluator.comparison_table(ci=True)
print('Classical ML Results (test set):')
table_ml

## 3. ROC & Precision-Recall Curves (Classical ML)

In [ ]:
viz = ResultsVisualizer(output_dir='../results/figures', dpi=150)
roc_data = evaluator.get_roc_data()

fig = viz.plot_roc_curves(roc_data, title='ROC Curves — Classical ML',
                          filename='nb_roc_classical')
plt.show()

y_probs_ml = {name: res['y_prob'] for name, res in evaluator.get_results().items()}
fig2 = viz.plot_pr_curves(y_test, y_probs_ml,
                          title='Precision-Recall — Classical ML',
                          filename='nb_pr_classical')
plt.show()

## 4. Feature Importance (Random Forest & XGBoost)

In [ ]:
for model_key in ['random_forest', 'xgboost', 'lightgbm']:
    if model_key not in ml_models:
        continue
    model = ml_models[model_key]
    imp_df = model.get_feature_importance()
    if imp_df is None:
        continue
    imp_df['feature'] = feature_names[:len(imp_df)]
    imp_df = imp_df.sort_values('importance_mean', ascending=False).head(15)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(imp_df['feature'][::-1], imp_df['importance_mean'][::-1],
            color='#4878CF', alpha=0.85)
    ax.set_title(f'Top 15 Feature Importances — {model.name}', fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

## 5. Deep Learning Models (MLP & TabNet)

In [ ]:
dl_evaluator = ModelEvaluator(bootstrap=True, n_bootstrap=300)

# MLP
print('Training MLP ...')
mlp = MLPModel(hidden_dims=[128, 64, 32], max_epochs=100, patience=15, random_state=SEED)
mlp.fit(X_train, y_train, X_val, y_val)
y_prob_mlp = mlp.predict_proba(X_test)[:, 1]
res_mlp = dl_evaluator.add('MLP', y_test, y_prob_mlp)
print(f'  MLP: AUROC={res_mlp["roc_auc"]:.3f}  AP={res_mlp["average_precision"]:.3f}')

# TabNet
try:
    print('Training TabNet ...')
    tabnet = TabNetModel(n_d=16, n_a=16, max_epochs=100, patience=15, random_state=SEED)
    tabnet.fit(X_train, y_train, X_val, y_val)
    y_prob_tab = tabnet.predict_proba(X_test)[:, 1]
    res_tab = dl_evaluator.add('TabNet', y_test, y_prob_tab)
    print(f'  TabNet: AUROC={res_tab["roc_auc"]:.3f}  AP={res_tab["average_precision"]:.3f}')
except Exception as e:
    print(f'  TabNet failed: {e}')

## 6. Ensemble Models

In [ ]:
from sklearn.base import clone

ens_evaluator = ModelEvaluator(bootstrap=True, n_bootstrap=300)

# Build base estimators from sklearn
base_ests = [
    ('LR',  ml_models['logistic_regression'].estimator),
    ('RF',  ml_models['random_forest'].estimator),
    ('XGB', ml_models['xgboost'].estimator),
]

# Soft Voting
print('Training Soft Voting ...')
voting = VotingEnsemble([(n, clone(e)) for n, e in base_ests], voting='soft')
voting.fit(X_tv, y_tv)
yp_vote = voting.predict_proba(X_test)[:, 1]
res_v = ens_evaluator.add('Soft Voting', y_test, yp_vote)
print(f'  Voting: AUROC={res_v["roc_auc"]:.3f}')

# Stacking
print('Training Stacking ...')
stacking = StackingEnsemble([(n, clone(e)) for n, e in base_ests], n_splits=5, random_state=SEED)
stacking.fit(X_tv, y_tv)
yp_stack = stacking.predict_proba(X_test)[:, 1]
res_s = ens_evaluator.add('Stacking', y_test, yp_stack)
print(f'  Stacking: AUROC={res_s["roc_auc"]:.3f}')

# Bayesian Averaging
print('Training Bayesian Averaging ...')
bma = BayesianModelAveraging([(n, clone(e)) for n, e in base_ests], random_state=SEED)
bma.fit(X_tv, y_tv)
yp_bma = bma.predict_proba(X_test)[:, 1]
res_bma = ens_evaluator.add('Bayesian Averaging', y_test, yp_bma)
print(f'  BMA: AUROC={res_bma["roc_auc"]:.3f}')

## 7. Comprehensive Comparison

In [ ]:
# Merge all evaluators for a unified comparison
all_results = {}
for name, res in evaluator.get_results().items():
    all_results[name] = res
for name, res in dl_evaluator.get_results().items():
    all_results[name] = res
for name, res in ens_evaluator.get_results().items():
    all_results[name] = res

# Summary table
metric_cols = ['roc_auc', 'average_precision', 'sensitivity', 'specificity', 'ppv', 'f1']
rows = []
for name, res in all_results.items():
    row = {'Model': name}
    for m in metric_cols:
        row[m] = round(res.get(m, float('nan')), 3)
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index('Model').sort_values('roc_auc', ascending=False)
print('All Models — Test Set Performance:')
comparison_df

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(10, max(5, len(comparison_df) * 0.5)))
sns.heatmap(comparison_df.astype(float), annot=True, fmt='.3f',
            cmap='YlOrRd', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Score'})
ax.set_title('Model Performance Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC overlay for all models
from sklearn.metrics import roc_curve
PALETTE_16 = plt.cm.tab20.colors

fig, ax = plt.subplots(figsize=(9, 7))
for i, (name, res) in enumerate(all_results.items()):
    fpr, tpr, _ = roc_curve(res['y_true'], res['y_prob'])
    ax.plot(fpr, tpr, lw=1.5, color=PALETTE_16[i % len(PALETTE_16)],
            label=f"{name} ({res['roc_auc']:.3f})")
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('1 − Specificity', fontsize=12)
ax.set_ylabel('Sensitivity', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=7.5, ncol=2)
plt.tight_layout()
plt.show()

## 8. Calibration Analysis

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0,1],[0,1],'k--', label='Perfect calibration')

top5 = list(all_results.items())[:5]
for i, (name, res) in enumerate(top5):
    frac_pos, mean_pred = calibration_curve(res['y_true'], res['y_prob'], n_bins=8)
    ax.plot(mean_pred, frac_pos, 's-', lw=2, label=name, color=PALETTE_16[i])

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves (Top 5 Models)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 9. SHAP Interpretability (Best Model)

In [ ]:
try:
    import shap
    
    # Use Random Forest for SHAP (TreeExplainer is fast)
    rf_model = ml_models['random_forest'].estimator
    
    explainer  = shap.TreeExplainer(rf_model)
    shap_vals  = explainer.shap_values(X_test[:200])
    shap_class1 = shap_vals[1] if isinstance(shap_vals, list) else shap_vals
    
    print('SHAP Summary Plot (Random Forest):')
    shap.summary_plot(shap_class1, X_test[:200], feature_names=feature_names,
                      max_display=15, show=True)
    
    print('\nSHAP Bar Plot:')
    shap.summary_plot(shap_class1, X_test[:200], feature_names=feature_names,
                      plot_type='bar', max_display=15, show=True)
except ImportError:
    print('Install shap for interpretability analysis: pip install shap')
except Exception as e:
    print(f'SHAP failed: {e}')

## 10. Summary

| Category | Best Model | AUROC |
|----------|-----------|-------|
| Classical ML | *(see table above)* | |
| Deep Learning | MLP / TabNet | |
| Ensemble | Stacking / BMA | |

**Key observations:**
- Ensemble methods generally outperform individual models
- Gradient boosting (XGBoost, LightGBM) are competitive with lower complexity
- Blood pressure and proteinuria are the strongest predictors per SHAP
- Clinical composite features (HELLP score, BP severity) add predictive value

---
*For complete pipeline with hyperparameter tuning, run the scripts in `pipelines/`.*